# Nexsure EAI API – Demo Notebook

End-to-end walkthrough:
1. Authenticate and get a bearer token
2. Discover branches and departments
3. Create a new client
4. Add a policy to that client
5. Attach a file to the policy
6. Read lookup data
7. Search clients
8. Search SIC / NAICS codes
9. Search policies with details
10. Search claims

In [1]:
import sys
sys.path.insert(0, '..')

from nexsure_api.nexsure_api_client import NexsureApiClient
from nexsure_api.credentials import NoAuth, NexsureCredentials
from nexsure_api.input_models import AddressInput, AssignmentInput, ContactInput
from nexsure_api.types import (
    ClientStage, ClientType, LegalEntity,
    LookupCategory,
    PolicyMode, PolicyStage, PolicyType,
    SearchType,
)

## 1. Authenticate

In [ ]:
INTEGRATION_KEY   = ""
INTEGRATION_LOGIN = ""
INTEGRATION_PWD   = ""

client = NexsureApiClient(credentials=[NoAuth()])

token_response = client.services.GetToken.execute(
    integration_key=INTEGRATION_KEY,
    integration_login=INTEGRATION_LOGIN,
    integration_pwd=INTEGRATION_PWD,
)

access_token = token_response.access_token
client.add_credentials(NexsureCredentials(api_token=access_token))
print(f"Token OK — expires in {token_response.expires_in // 3600}h")

Token OK — expires in 3h


## 2. Discover branches and departments

In [3]:
branches = client.services.SearchBranchByBranchName.execute()
for b in branches.Branch:
    print(f"  branch  id={b.BranchID:<4} active={b.IsActive}  {b.BranchName}")

departments = client.services.SearchDepartmentByName.execute()
for d in departments.Department:
    print(f"  dept    id={d.DepartmentID:<4} active={d.IsActive}  {d.DepartmentName}")

branch_id     = branches.Branch[0].BranchID
department_id = next(d.DepartmentID for d in departments.Department if d.IsActive == "true")
assignment    = AssignmentInput(branch_id=branch_id, department_id=department_id)
print(f"\nUsing branch={branch_id}  department={department_id}")

  branch  id=1    active=true  Insurekore Ltd.
  branch  id=2    active=false  Tech-Insure Consulting Ltd.
  branch  id=3    active=true  Tech-Insure Solutions Ltd.
  branch  id=4    active=true  Wetraq
  branch  id=5    active=true  Test Branch
  branch  id=6    active=true  Semsee Corp.
  dept    id=0    active=false  Unassigned
  dept    id=1    active=true  CA$
  dept    id=2    active=true  US$
  dept    id=3    active=true  Personal Lines
  dept    id=4    active=true  Test Department
  dept    id=5    active=true  Commercial Lines

Using branch=1  department=1


## 3. Create a new client

In [4]:
add_client_response = client.services.AddNewClient.execute(
    name="Acme Corporation",
    assignment=assignment,
    client_type=ClientType.Commercial,
    stage=ClientStage.Prospect,
    legal_entity=LegalEntity.Corporation,
    contacts=[ContactInput(first_name="John", last_name="Doe")],
    addresses=[AddressInput(street="123 Main St", city="Chicago", state="IL", zip_code="60601")],
)

new_client_id = add_client_response.Client.ClientID
print(f"Created client  ID={new_client_id}")

Created client  ID=5658


## 4. Add a policy

`PolicyStage.Marketing` creates a pre-in-force marketing record — no carrier required.

In [5]:
add_policy_response = client.services.AddSinglePolicy.execute(
    client_id=new_client_id,
    policy_number="POL-2024-001",
    assignment=assignment,
    eff_date="2024-01-01",
    exp_date="2025-01-01",
    description="General Liability",
    mode=PolicyMode.New,
    stage=PolicyStage.Marketing,
    policy_type=PolicyType.Monoline,
)

new_policy_id = add_policy_response.Policy.PolicyID
print(f"Created policy  ID={new_policy_id}  stage={add_policy_response.Policy.PolicyStage}")

Created policy  ID=5274  stage=Marketing


## 5. Attach a file

Pass `file_bytes=` for real content; omit it to register a placeholder record.

In [6]:
add_attachment_response = client.services.AddAttachment.execute(
    client_id=new_client_id,
    policy_id=new_policy_id,
    file_name="declaration_page.pdf",
    description="Policy declaration page",
)

att = add_attachment_response.Attachment
print(f"Attachment ID={att.AttachmentID}  file={att.FileName}  modified={att.LastModifiedDt}")

Attachment ID=4058  file=declaration_page.pdf  modified=2026-04-23T20:32:16.0436828-07:00


## 6. Read lookup data

Valid `LookupCategory` values: `AdditionalInterest`, `Carrier`, `Client`, `DocumentIntegration`,
`FinancialEntity`, `Miscellaneous`, `Organization`, `People`, `Policy`,
`PremiumFinanceCompany`, `RetailAgent`, `TaxAuthority`, `Vendor`

In [7]:
lookup_response = client.services.ListLookupManagementValues.execute(
    category_name=LookupCategory.Policy,
)

for category in lookup_response.Category:
    print(f"Category: {category.CategoryName}")
    for lookup_type in category.Type:
        print(f"  {lookup_type.TypeName} ({len(lookup_type.DataItem)} items)")
        for item in lookup_type.DataItem[:3]:
            print(f"    {item.ItemValue}")

Category: Policy
  Market Analysis Reasons (4 items)
    Client Preferred Carrier
    Client Preferred Coverage
    Client Preferred Premium
  Market Analysis Statuses (2 items)
    Finalized
    Pending
  Claim Stages (1 items)
    Subrogation                                       
  Claim Types (0 items)
  Code Classes (0 items)
  Binder Term (1 items)
    30 Days   
  Claim Payment Types (16 items)
    Adjustment Expense
    All
    BI Liab
  Code Designation (0 items)


## 7. Search clients

In [8]:
client_list = client.services.GetClientList.execute(client_name="a ")

print(f"{'ID':<8} {'Name':<35} {'Type':<12} {'Stage':<12} {'City, State'}")
print("-" * 90)
for c in client_list.clients[:10]:
    print(f"{c.ClientId:<8} {c.ClientName:<35} {c.ClientType:<12} {c.ClientStage:<12} {c.LocCity}, {c.LocState}")

ID       Name                                Type         Stage        City, State
------------------------------------------------------------------------------------------
833      02.08.24 12.00 Autotests check name trimmedMagna dolore nonu Commercial   Client       Beverly Hills, CA
817      02.08.24 Autotests check name trimmedMagna dolore ut amet al Commercial   Client       Beverly Hills, CA
866      02.09.24 12.00 Autotests check name trimmedErat magna nibh n Commercial   Client       Beverly Hills, CA
968      02.28.24 12.00 Autotests check name trimmedLaoreet magna lao Commercial   Client       Beverly Hills, CA
1000     03.01.24 12.00 Autotests check name trimmedElit sed magna do Commercial   Client       Beverly Hills, CA
1082     12QA Stick Stickly Test             Commercial   Client       NEW YORK, NY
3401     A Stitch Above Embroidery           Commercial   Prospect     Brighton, MI
448      Aishwarya sharma                    Personal     Prospect     New York, NY
471 

## 8. Search SIC / NAICS codes

In [9]:
sic_response = client.services.SicNaicsSearch.execute(
    naics_description="",
    search_type=SearchType.Contains,
    results_per_page=10,
)

print(f"Page 1 of {sic_response.total_pages}  ({len(sic_response.codes)} codes shown)")
print(f"{'NAICS':<12} {'SIC':<8} Description")
print("-" * 70)
for code in sic_response.codes:
    print(f"{(code.NaicsCode or ''):<12} {(code.SicCode or ''):<8} {code.NaicsDescription or code.SicDescription}")

Page 1 of 403  (10 codes shown)
NAICS        SIC      Description
----------------------------------------------------------------------
111                   [2001] Crop Production
1111                  [2001] Oilseed and Grain Farming
11119                 [2001] Other Grain Farming
1112                  [2001] Vegetable and Melon Farming
11121                 [2001] Vegetable and Melon Farming
1113                  [2001] Fruit and Tree Nut Farming
11133                 [2001] Noncitrus Fruit and Tree Nut
1114                  [2001] Greenhouse, Nursery, and
11141                 [2001] Food Crops Grown Under Cover
11142                 [2001] Nursery and Floriculture Production


## 9. Search policies with details

In [10]:
policy_search = client.services.PolicySearchWithDetails.execute(
    client_name="",
    search_type=SearchType.Contains,
    results_per_page=10,
)

print(f"Page 1 of {policy_search.total_pages}  ({len(policy_search.policies)} policies shown)")
print(f"{'PolicyID':<10} {'Number':<22} {'Stage':<14} {'Status':<12} {'Eff':<12} Exp")
print("-" * 90)
for p in policy_search.policies:
    print(f"{(p.PolicyID or ''):<10} {(p.PolicyNumber or ''):<22} {(p.PolicyStage or ''):<14} {(p.PolicyStatus or ''):<12} {(p.EffDate or ''):<12} {p.ExpDate or ''}")

Page 1 of 304  (10 policies shown)
PolicyID   Number                 Stage          Status       Eff          Exp
------------------------------------------------------------------------------------------
84                                Marketing      Pending      2020-04-27   2021-04-27
85                                Marketing      Pending      2020-04-27   2021-04-27
86                                Marketing      Pending      2016-06-12   2017-06-12
109                               Marketing      Pending      2022-11-08   2023-11-08
110                               Marketing      Pending      2022-11-08   2023-11-08
111                               Marketing      Pending      2022-11-08   2023-11-08
114                               Marketing      Pending      2022-11-08   2023-11-08
4614       00-abcd-efgh           Marketing      Pending      2025-05-09   2026-05-09
4615       00-abcd-efgh           Marketing      Pending      2025-05-09   2026-05-09
4616       00-abcd-ef

## 10. Search claims

Filter by `client_name`, `claim_number`, `policy_number`, `claimant_name`,
`adjustor_name`, `date_of_loss_from`/`date_of_loss_to`.

In [11]:
claim_search = client.services.ClaimSearch.execute(
    search_type=SearchType.Contains,
    results_per_page=10,
)

print(f"Page 1 of {claim_search.total_pages}  ({len(claim_search.claims)} claims shown)")
print(f"{'ClaimID':<10} {'Date of Loss':<14} {'Status':<10} {'Stage':<10} {'Est. Amount':<14} Paid")
print("-" * 80)
for claim in claim_search.claims:
    d = claim.ClaimDetail
    print(f"{(claim.ClaimID or ''):<10} {(d.DateOfLoss or '') if d else '':<14} "
          f"{(d.Status or '') if d else '':<10} {(d.Stage or '') if d else '':<10} "
          f"{(d.EstimatedAmount or '') if d else '':<14} {(d.TotalPaidAmount or '') if d else ''}")

Page 1 of 4  (10 claims shown)
ClaimID    Date of Loss   Status     Stage      Est. Amount    Paid
--------------------------------------------------------------------------------
42         2024-06-12     Closed                               
15         2024-04-30     Open                                 
16         2024-04-30     Open                                 
17         2024-04-30     Open                                 
47         2025-07-09     Open                                 
41         2024-06-07     Open                                 
34         2024-05-16     Closed                               
35         2024-05-20     Closed                               
24         2024-05-11     Open                                 
25         2024-05-11     Open                                 
